### Por que testar?

 - Imagine que ao evoluir o projeto uma função muda (a build_features_base() por exemplo). Sem querer, quebra o encoding de Contract. O modelo re-treina, os números ficam parecidos e você nem percebe. Em produção, a performance cai drasticamente.


 - É pra isso que Testes servem. Ele é basicamente um contrato.. Você escreve uma função, e o teste diz "Essa função SEMPRE deve se comportar assim". Se alguém mudar a função o teste GRITA imediantamente.

In [42]:
def soma(a, b):
    return a + b

resultado = soma(2,3)
if resultado !=5:
    raise AssertionError(f'Esperado 5, mas obteve {resultado}')
print('OK - Teste passou com sucesso!')

OK - Teste passou com sucesso!


In [ ]:
def calcula_taxa_churn(n_churn, n_total):
    if n_total == 0:
        raise ValueError('O número total de clientes não pode ser zero.')
    if n_churn < 0 or n_total < 0:
        raise ValueError('O número de clientes churn ou total não pode ser negativo.')
    print('OK - Teste passou com sucesso!')
    return n_churn / n_total

resultado = calcula_taxa_churn(10, 100)
if resultado != 0.1:
    raise AssertionError(f'Esperado 0.1, mas obteve {resultado}')



OK - Teste passou com sucesso!


### Asserts

- Basicamente o Python tem uma palavra chave chamada 'assert' que faz exatamente o que o if / raise faz.

In [44]:
def calcula_taxa_churn(n_churn, n_total):
    return n_churn / n_total

In [59]:
assert calcula_taxa_churn(10, 100) == 0.1, 'Teste falhou: taxa de churn calculada incorretamente.'
assert calcula_taxa_churn(-10, 100) != -0.1, 'Teste falhou: taxa de churn calculada incorretamente para valores negativos.'

AssertionError: Teste falhou: taxa de churn calculada incorretamente para valores negativos.

### Pytest

- O Pytest descrobre e executa todos os testes do projeto com um único comando, e apresenta um relatório detalhado dos resultados..

- Comando: pytest tests/

- Para fazer corretamente deve respeitar algumas convenções:

- O nome da função de teste deve começar com "test_".
- O teste deve ser independente, ou seja, não deve depender de outros testes.
- Usa assert para verificar se o resultado esperado é igual ao resultado obtido.

In [46]:
# Exemplo

def encode_contract(valor):
    mapa = {"Month-to-month": 0, "One year": 1, "Two year": 2}
    return mapa[valor]

In [47]:
def test_contract_mensal():
    assert encode_contract("Month-to-month") == 0

def test_contract_anual():
    assert encode_contract("One year") == 1

def test_contract_bienal():
    assert encode_contract("Two year") == 2

In [48]:
test_contract_mensal()
test_contract_anual()
test_contract_bienal()
print("Todos os testes passaram.")

Todos os testes passaram.


### Fixtures: Dados de teste reutilizaveis


In [49]:
import pandas as pd
pd.set_option('display.max_columns', None)

# Isso é uma fixture (no pytest será decorada com @pytest.fixture)
def sample_raw_data():
    return pd.DataFrame({
        "id":             [1, 2],
        "gender":         ["Male", "Female"],
        "SeniorCitizen":  [0, 1],
        "Partner":        ["Yes", "No"],
        "Dependents":     ["No", "Yes"],
        "tenure":         [12, 48],
        "PhoneService":   ["Yes", "Yes"],
        "MultipleLines":  ["No", "Yes"],
        "InternetService":["Fiber optic", "DSL"],
        "OnlineSecurity": ["No", "Yes"],
        "OnlineBackup":   ["No", "No"],
        "DeviceProtection":["No", "Yes"],
        "TechSupport":    ["No", "No"],
        "StreamingTV":    ["Yes", "No"],
        "StreamingMovies":["Yes", "No"],
        "Contract":       ["Month-to-month", "Two year"],
        "PaperlessBilling":["Yes", "No"],
        "PaymentMethod":  ["Electronic check", "Bank transfer (automatic)"],
        "MonthlyCharges": [70.35, 20.15],
        "TotalCharges":   [844.2, 967.2],
        "Churn":          ["Yes", "No"],
    })

In [50]:
df = sample_raw_data()

In [51]:
import sys
import os

# Adiciona o diretório raiz do projeto ao sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [52]:
from src.features.build import build_features_base

In [57]:
# IsAutoPayment é 0 para "Electronic check" e 1 para "Bank transfer (automatic)"
def test_auto_payment():
    df = sample_raw_data()
    df = build_features_base(df)
    assert df.loc[0, "IsAutoPayment"] == 0, f'Esperado 0 para IsAutoPayment do cliente 1, mas obteve {df.loc[0, "IsAutoPayment"]}'
    assert df.loc[1, "IsAutoPayment"] == 1, f'Esperado 1 para IsAutoPayment do cliente 2, mas obteve {df.loc[1, "IsAutoPayment"]}'

# Contract do cliente 1 é 0 (Month-to-month)
def test_contract_cliente_1():
    df = sample_raw_data()
    df = build_features_base(df)
    assert df.loc[0, "Contract"] == 0, f'Esperado 0 para Contract do cliente 1, mas obteve {df.loc[0, "Contract"]}'

# PaymentMethod não existe mais no DataFrame após o processamento
def test_payment_method_removido():
    df = sample_raw_data()
    df = build_features_base(df)
    assert "PaymentMethod" not in df.columns, 'Teste falhou: PaymentMethod ainda existe no DataFrame após o processamento.'

# ServicesBundle do cliente 1 é a soma correta dos seus 6 serviços
def test_services_bundle_cliente_1():
    df = sample_raw_data()
    df = build_features_base(df)
    expected_bundle = 4  # O cliente 1 tem PhoneService, StreamingTV, StreamingMovies e MultipleLines
    assert df.loc[0, "ServicesBundle"] == expected_bundle, f'Esperado {expected_bundle} para ServicesBundle do cliente 1, mas obteve {df.loc[0, "ServicesBundle"]}'

In [58]:
test_auto_payment()
test_contract_cliente_1()
test_payment_method_removido()
test_services_bundle_cliente_1()
print("Todos os testes passaram.")

AssertionError: Esperado 4 para ServicesBundle do cliente 1, mas obteve 2